In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 11:57:55 WARN Utils: Your hostname, Khanhs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.50.36 instead (on interface en0)
26/03/09 11:57:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 11:57:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 11:57:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark.version

'4.1.1'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 12:01:46--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.224.173.162, 13.224.173.219, 13.224.173.98, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.224.173.162|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  5.61MB/s    in 11s     

2026-03-09 12:01:57 (6.46 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [45]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
df_4=df.repartition(4)

In [7]:
df_4.write.mode("overwrite").parquet("output/yellow_nov_2025_4parts")

In [46]:
df = df \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [47]:
from pyspark.sql import functions as F

In [48]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [49]:
df_filter_15nov = df.filter(
    (df.pickup_datetime >= "2025-11-15") & 
    (df.pickup_datetime < "2025-11-16")
)

In [50]:
df_filter_15nov.count()

162604

In [51]:
df_trip_length = df.withColumn(
    "trip_length", 
    (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"))/3600
)

In [52]:
from pyspark.sql.functions import desc

df_trip_length.select("trip_length").orderBy(desc("trip_length")).show()

[Stage 18:==========================================>              (9 + 3) / 12]

+------------------+
|       trip_length|
+------------------+
| 90.64666666666666|
| 76.94833333333334|
| 76.21388888888889|
| 69.28861111111111|
| 67.08055555555555|
| 63.36833333333333|
|56.382222222222225|
|48.147777777777776|
| 47.47833333333333|
| 45.44416666666667|
| 44.04333333333334|
|43.230555555555554|
|42.720555555555556|
|41.614444444444445|
| 41.50333333333333|
| 39.33305555555555|
|38.074444444444445|
| 37.91111111111111|
| 34.87027777777778|
| 30.53388888888889|
+------------------+
only showing top 20 rows


In [21]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 14:05:55--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.224.173.98, 13.224.173.162, 13.224.173.176, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.224.173.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 14:05:55 (905 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [30]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [31]:
df_zones.schema

StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [32]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [33]:
df_zones.createOrReplaceTempView("zones")

In [38]:
df_zones = df_zones.withColumn("LocationID", F.col("LocationID").cast("int"))

In [54]:
df.createOrReplaceTempView('trips_data')

In [57]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS trip_count
FROM 
    trips_data t
JOIN zones z
    ON t.PULocationID = z.LocationID
GROUP BY 
    z.Zone
ORDER BY
    trip_count ASC
LIMIT 10
""").show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
|Rossville/Woodrow                            |4         |
|Great Kills                                  |4         |
|Green-Wood Cemetery                          |4         |
|Jamaica Bay                                  |5         |
|Westerleigh                                  |12        |
+---------------------------------------------+----------+



In [111]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')